# ARC3 Agent：从零看懂

欢迎！这个 notebook 是整个项目的起点。

我们会用**可执行代码 + 源码查看**的方式，一步步理解这个 ARC-AGI-3 轻量研究脚手架的核心思想：

1. **环境** (Env) —— agent 看到什么、能做什么
2. **世界模型** (World Model) —— 会“想象”未来网格的 Dreamer-lite
3. **规划 Agent** —— 用想象 rollout 选择动作
4. **自调优平台** (PBT) —— 自动调参的训练框架
5. **提交路径** —— 极小的离线 inference 代码

所有代码都直接 import 自本项目真实实现 (`arcagent.*` 和 `submission.*`)。


## 1. 快速感受整个系统

In [ ]:
from arcagent.envs import MockArcEnv
from arcagent.agent import WorldModelAgent

env = MockArcEnv(grid_size=16, max_steps=30, seed=42)
obs = env.reset()
print("Observation keys:", list(obs.keys()))
print("Grid shape:", obs["grid"].shape)
print("Available actions:", obs["available_actions"])


颜色 0-15 代表不同的 ARC 网格颜色。Agent 的核心循环是：

- 观察当前 grid
- 在“头脑”里想象执行不同动作后可能出现的未来网格
- 挑选预测累计奖励最高的动作

下面我们直接实例化一个会想象的 agent。

In [ ]:
agent = WorldModelAgent.create(grid_size=16, seed=0)
print("Agent type:", type(agent).__name__)
print("World model type:", type(agent.world_model).__name__)


## 2. 核心对象一览（直接看源码！）

In [ ]:
import inspect
from arcagent.agent.world_model_agent import WorldModelAgent as WMA
from arcagent.models.dreamer_lite import DreamerLiteWorldModel

print("=== WorldModelAgent.act 核心逻辑 ===")
print(inspect.getsource(WMA.act)[:1200])


## 3. 世界模型做了什么？

In [ ]:
print("=== DreamerLiteWorldModel 关键方法 ===")
print(inspect.getsource(DreamerLiteWorldModel.encode)[:600])
print("\n...\n")
print(inspect.getsource(DreamerLiteWorldModel.predict_next_latent)[:600])


## 4. 接下来做什么？

继续下一个 notebook 开始**深入环境**：

→ `02_environment_explorer.ipynb`

或者直接跳到你感兴趣的部分：

- 想看想象 rollout → 04_planning_and_imagination.ipynb
- 想看自动调参 → 05_self_tuning_platform.ipynb
- 想跑真实训练再看结果 → 先跑 `python -m arcagent.train.run --steps 200`，然后打开 06

所有 notebook 都支持 **marimo** 和 Jupyter。


## 5. 小实验

试着改下面的 grid_size 或 steps，看看想象出来的网格形状变化。

In [ ]:
imagined_grids = agent.imagined_rollout(obs['grid'], steps=5, action=0)
print("Imagined grids shapes:", [g.shape for g in imagined_grids])
print("Last imagined grid (first 6x6):")
print(imagined_grids[-1][:6, :6])
